# Vertical-Well Blowdown — N₂, Godunov/HLLC Finite Volume (Phase 3, Stage 1)

Adiabatic blowdown of a 2,000-ft (609.6 m) vertical well, initially at
13.79 MPa / 300 K, closed bottom, discharging through a 12.7 mm wellhead
choke (Cₘ = 0.84) to atmosphere. Formulation: `docs/wellbore_model.md`.

The wellhead BC couples the interior solution (C⁺ characteristic +
isentrope) to an isentropic-nozzle choke model with automatic
choked/unchoked detection; mass and total-energy (incl. potential)
budgets close to machine precision.

In [ ]:
try:
    import hyperbolic1d
except ImportError:
    %pip install -q "hyperbolic1d[plot] @ git+https://github.com/JaderBarbosaLSU/hyperco2-wellbore"
import numpy as np, matplotlib.pyplot as plt
from hyperbolic1d.eos import NITROGEN
from hyperbolic1d.wellbore import (WellboreModel, WellGeometry, ChokeValve,
                                   vessel_blowdown_ode)

## 1. Model setup and run

In [ ]:
geom  = WellGeometry(depth=609.6, diameter=0.100)
choke = ChokeValve(diameter=0.0127, cd=0.84, p_back=101325.0)
model = WellboreModel(geometry=geom, choke=choke, eos=NITROGEN,
                      gravity=True, n_cells=300)
P0, T0, T_END = 1.379e7, 300.0, 600.0   # 2,000 psi, 300 K, 10 min
res = model.solve(P0, T0, T_END, n_snapshots=7)
print(f"{res.n_steps} steps in {res.wall_time:.0f} s;"
      f" mass budget {res.mass_balance_error:.1e},"
      f" energy budget {res.energy_balance_error:.1e}")

## 2. Profiles p, T, ρ, u along the well

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
cm = plt.cm.viridis
for i, ts in enumerate(res.snap_times):
    c = cm(i / max(len(res.snap_times) - 1, 1)); kw = dict(color=c, lw=1.2, label=f"t = {ts:.0f} s")
    axes[0,0].plot(res.z, res.p[i]/1e6, **kw); axes[0,1].plot(res.z, res.T[i], **kw)
    axes[1,0].plot(res.z, res.rho[i], **kw);   axes[1,1].plot(res.z, res.u[i], **kw)
for ax, lab in zip(axes.flat, ["Pressure [MPa]","Temperature [K]","Density [kg/m$^3$]","Velocity [m/s]"]):
    ax.set_xlabel("z [m] (0 = bottom)"); ax.set_ylabel(lab); ax.grid(alpha=0.3)
axes[0,0].legend(fontsize=7); fig.tight_layout()

## 3. Wellhead histories and verification vs the 0-D vessel limit

In [ ]:
ref = vessel_blowdown_ode(NITROGEN, geom.volume, choke, P0, T0, T_END)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].plot(res.t_hist, res.mdot, label="1-D FV");      axes[0].plot(ref["t"], ref["mdot"], "k--", lw=1, label="0-D vessel ODE")
axes[0].set_ylabel("Wellhead $\\dot m$ [kg/s]"); axes[0].legend(fontsize=8)
axes[1].plot(res.t_hist, res.mass);  axes[1].plot(ref["t"], ref["mass"], "k--", lw=1)
axes[1].set_ylabel("Mass in well [kg]")
axes[2].plot(res.t_hist, res.T_wellhead); axes[2].set_ylabel("Wellhead T [K]")
for ax in axes: ax.set_xlabel("t [s]"); ax.grid(alpha=0.3)
fig.tight_layout()
# NOTE: the 0-D reference neglects gravity; mass agreement within ~1%
# is the quasi-static check. T reaches ~127 K by 600 s -- at N2's critical
# temperature: the motivation for the real-fluid EOS stage.

## 4. Experiments

Things to vary: `choke.diameter` (1/4 in ... open pipe), `gravity`,
`friction=True`, `init="hydrostatic"`, `wall_model=PrescribedWallTemperature(T_wall=300.0)`
(simplified heat-transfer benchmark), `n_cells`, `cfl`.

In [ ]:
# example: effect of wall heat transfer at fixed wall temperature
from hyperbolic1d.wellbore import PrescribedWallTemperature
m2 = WellboreModel(geometry=geom, choke=choke, gravity=True, n_cells=300,
                   wall_model=PrescribedWallTemperature(T_wall=300.0))
r2 = m2.solve(P0, T0, 300.0, n_snapshots=2)
plt.figure(figsize=(7,4))
plt.plot(res.t_hist, res.T_wellhead, label="adiabatic")
plt.plot(r2.t_hist, r2.T_wellhead, label="$T_w$ = 300 K (Dittus-Boelter)")
plt.xlabel("t [s]"); plt.ylabel("Wellhead T [K]"); plt.xlim(0, 300)
plt.grid(alpha=0.3); plt.legend();